In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import module, secret_keys
from model_list import models
import pandas as pd

hf_api_key             = secret_keys.HF_TOKEN                   #<insert your own huggingface token here>
openai_api_key         = secret_keys.OPENAI_API_KEY_TEAM        #<insert your own openai token here>

In [3]:
data_repo_eval = pd.read_csv('hidden_data/CT-Repo-With-Examples-Corrected-allgpteval.csv')
data_repo_gen = pd.read_csv('hidden_data/CT-Repo-With-Examples-Corrected-allgen.csv')
print(data_repo_eval.shape)
print(data_repo_gen.shape)

(1693, 5)
(1693, 14)


In [4]:
data_repo_eval.head(2)

,NCTId,gpt4o_zs_gen_matches,gpt4o_ts_gen_matches,llama3_70b_it_zs_gen_matches,llama3_70b_it_ts_gen_matches
0,NCT00000620,NaN,NaN,NaN,NaN
1,NCT00003901,"{\n ""matched_features"": [\n [""Age, C...","{\n ""matched_features"": [\n [""Age, C...","{\n ""matched_features"": [\n [""Age, C...","{\n ""matched_features"": [\n [""Age, C..."


In [ ]:
data_repo_gen.head(2)

,NCTId,BriefTitle,EligibilityCriteria,BriefSummary,Conditions,Interventions,PrimaryOutcomes,TrialGroup,API_BaselineMeasures,API_BaselineMeasures_Corrected,gpt4o_zs_gen,gpt4o_ts_gen,llama3_70b_it_zs_gen,llama3_70b_it_ts_gen
0,NCT00000620,Action to Control Cardiovascular Risk in Diabe...,Inclusion Criteria:\n\n* Diagnosed with type 2...,The purpose of this study is to prevent major ...,"Atherosclerosis, Cardiovascular Diseases, Hype...","Anti-hyperglycemic Agents, Anti-hypertensive A...",First Occurrence of a Major Cardiovascular Eve...,hypertension,"Age, Continuous, Gender, Ethnicity (NIH/OMB), ...","`Age, Continuous`, `Gender`, `Ethnicity (NIH/O...","`Age`, `Sex`, `Race/Ethnicity`, `Body Mass Ind...","`Age, Continuous`, `Gender`, `Ethnicity (NIH/O...","`Age`, `Sex`, `Race`, `Ethnicity`, `Body Mass ...","`Age, Continuous`, `Gender`, `Ethnicity (NIH/O..."
1,NCT00003901,Prognostic Study of Metastases in Patients Wit...,Inclusion Criteria:\n\n1. Patient must be ≥ 18...,RATIONALE: Prognostic testing for early signs ...,"Lung Cancer,","immunohistochemistry staining method, biopsy, ...",Overall Survival in Lymph Nodes Examined Patie...,cancer,"Age, Continuous, Gender, Race/Ethnicity, Custo...","`Age, Continuous`, `Gender`, `Race/Ethnicity, ...","`Age`, `Gender`, `ECOG/Zubrod status`, `Clinic...","`Age, Continuous`, `Sex: Female, Male`, `Race/...","`age`, `sex`, `ECOG/Zubrod status`, `histologi...","`Age, Continuous`, `Sex: Female, Male`, `Race/..."


## Example Hallucination Calculation Check 

In [ ]:

#`Gender` in reference and `Inflammation` in candidate are Negative hallucinations, not reported in matches or remainings 
reference_features = ['`Age`', '`Blood Pressure`', '`Height`', '`Gender`', '`Previous Medication`', '`Race`', '`Ethnicity`']
candidate_features = ['`Age`', '`Systolic Blood Pressure`', '`Diastolic Blood Pressure`', '`Body Mass Index`', '`Race`', "`Inflammation`"]

matched_results =   {
                        "matched_features": [
                            ["`Age`", "`Age`"],
                            ["`bogus1`", "`bogus 2"], ## <-- positive hallucinations in both reference and candidate, but we count only 1 for each matched pair
                            ["`body mass index`", "`Body Mass Index`"], ## <-- positive hallucination 'body mass index' doesn't exist in reference
                            ["`Blood Pressure`", "`Systolic Blood Pressure`"], ## <-- multimatch hallucination in reference (counting one of multiple matches as correct match)
                            ["`Blood Pressure`", "`Diastolic Blood Pressure`"],## <-- multimatch hallucination in reference
                            ["`Race`","`Race`"], ## <-- multimatch hallucination in candidate (counting one of multiple matches as correct match)
                            ["`Ethnicity`", "`Race`"], ## <-- multimatch hallucination in candidate
                            ["`Height`", "`patient height`"] ##<-- positive hallucination 'patient height' doesn't exist in candidate
                        ],
                        "remaining_reference_features": ["`Previous Medication`"],
                        "remaining_candidate_features": []
                    }

module.calculate_hallucination(reference_features, candidate_features, matched_results)

(3, 2, 2, 3)

## Calculate for whole CT-Pub Dataset and save in dataframe

In [7]:
repo_hallucination_results = pd.DataFrame()
repo_hallucination_results['NCTId'] = data_repo_gen['NCTId']
repo_hallucination_results['TrialGroup'] = data_repo_gen['TrialGroup']

In [8]:
import json 
ref_column_name = 'API_BaselineMeasures_Corrected'

for index, row_gen in data_repo_gen.iterrows():

    avoid_ids = ['NCT00000620', 'NCT01483560', 'NCT04280783'] #these were used as examples for 3-shot generation
    if row_gen['NCTId'] in avoid_ids:
        continue

    row_eval = data_repo_eval[data_repo_eval['NCTId'] == row_gen['NCTId']]
    if row_eval.empty:
        print(f"Missing NCTId {row_gen['NCTId']} in data_repo_eval")
        continue
    reference_features = module.extract_elements_v2(row_gen[ref_column_name])

    #calculate adjusted precision, recall and f1 for GPT4 zero shot generation
    gzs_candidate = module.extract_elements_v2(row_gen['gpt4o_zs_gen'])
    gzs_matches = json.loads(row_eval['gpt4o_zs_gen_matches'].values[0])
    gzs_hallucination = module.calculate_hallucination(reference_features, gzs_candidate, gzs_matches)
    if 'gpt4o_zs_gen_hal' not in repo_hallucination_results.columns:
        repo_hallucination_results['gpt4o_zs_gen_hal'] = None
    gzs_precision = gzs_hallucination[3]/len(gzs_candidate)
    gzs_recall = gzs_hallucination[3]/len(reference_features)
    gzs_f1 = 2 * (gzs_precision * gzs_recall) / (gzs_precision + gzs_recall) if gzs_precision + gzs_recall > 0 else 0
    repo_hallucination_results.at[index, 'gpt4o_zs_gen_hal'] = (gzs_hallucination[0], gzs_hallucination[1], gzs_hallucination[2], gzs_hallucination[3], gzs_precision, gzs_recall, gzs_f1)

    #calculate adjusted precision, recall and f1 for GPT4 three shot generation
    gts_candidate = module.extract_elements_v2(row_gen['gpt4o_ts_gen'])
    gts_matches = json.loads(row_eval['gpt4o_ts_gen_matches'].values[0])
    gts_hallucination = module.calculate_hallucination(reference_features, gts_candidate, gts_matches)
    if 'gpt4o_ts_gen_hal' not in repo_hallucination_results.columns:
        repo_hallucination_results['gpt4o_ts_gen_hal'] = None
    gts_precision = gts_hallucination[3]/len(gts_candidate)
    gts_recall = gts_hallucination[3]/len(reference_features)
    gts_f1 = 2 * (gts_precision * gts_recall) / (gts_precision + gts_recall) if gts_precision + gts_recall > 0 else 0
    repo_hallucination_results.at[index, 'gpt4o_ts_gen_hal'] = (gts_hallucination[0], gts_hallucination[1], gts_hallucination[2], gts_hallucination[3], gts_precision, gts_recall, gts_f1)

    #calculate adjusted precision, recall and f1 for LLAMA3 zero shot generation
    lzs_candidate = module.extract_elements_v2(row_gen['llama3_70b_it_zs_gen'])
    lzs_matches = json.loads(row_eval['llama3_70b_it_zs_gen_matches'].values[0])
    lzs_hallucination = module.calculate_hallucination(reference_features, lzs_candidate, lzs_matches)
    if 'llama3_70b_it_zs_gen_hal' not in repo_hallucination_results.columns:
        repo_hallucination_results['llama3_70b_it_zs_gen_hal'] = None
    lzs_precision = lzs_hallucination[3]/len(lzs_candidate)
    lzs_recall = lzs_hallucination[3]/len(reference_features)
    lzs_f1 = 2 * (lzs_precision * lzs_recall) / (lzs_precision + lzs_recall) if lzs_precision + lzs_recall > 0 else 0
    repo_hallucination_results.at[index, 'llama3_70b_it_zs_gen_hal'] = (lzs_hallucination[0], lzs_hallucination[1], lzs_hallucination[2], lzs_hallucination[3], lzs_precision, lzs_recall, lzs_f1)

    #calculate adjusted precision, recall and f1 for LLAMA3 three shot generation
    lts_candidate = module.extract_elements_v2(row_gen['llama3_70b_it_ts_gen'])
    lts_matches = json.loads(row_eval['llama3_70b_it_ts_gen_matches'].values[0])
    lts_hallucination = module.calculate_hallucination(reference_features, lts_candidate, lts_matches)
    if 'llama3_70b_it_ts_gen_hal' not in repo_hallucination_results.columns:
        repo_hallucination_results['llama3_70b_it_ts_gen_hal'] = None
    lts_precision = lts_hallucination[3]/len(lts_candidate)
    lts_recall = lts_hallucination[3]/len(reference_features)
    lts_f1 = 2 * (lts_precision * lts_recall) / (lts_precision + lts_recall) if lts_precision + lts_recall > 0 else 0
    repo_hallucination_results.at[index, 'llama3_70b_it_ts_gen_hal'] = (lts_hallucination[0], lts_hallucination[1], lts_hallucination[2], lts_hallucination[3], lts_precision, lts_recall, lts_f1)


In [9]:
repo_hallucination_results

,NCTId,TrialGroup,gpt4o_zs_gen_hal,gpt4o_ts_gen_hal,llama3_70b_it_zs_gen_hal,llama3_70b_it_ts_gen_hal
0,NCT00000620,hypertension,None,None,None,None
1,NCT00003901,cancer,"(0, 0, 0, 4, 0.2222222222222222, 0.36363636363...","(0, 0, 0, 7, 0.5833333333333334, 0.63636363636...","(0, 0, 0, 5, 0.45454545454545453, 0.4545454545...","(0, 0, 0, 7, 0.6363636363636364, 0.63636363636..."
2,NCT00005879,cancer,"(0, 0, 0, 4, 0.125, 0.3333333333333333, 0.1818...","(0, 0, 0, 6, 0.2857142857142857, 0.5, 0.363636...","(0, 0, 0, 5, 0.25, 0.4166666666666667, 0.3125)","(0, 0, 0, 6, 0.2608695652173913, 0.5, 0.342857..."
3,NCT00005908,cancer,"(0, 0, 0, 2, 0.125, 0.3333333333333333, 0.1818...","(0, 0, 0, 4, 0.2857142857142857, 0.66666666666...","(0, 0, 0, 4, 0.23529411764705882, 0.6666666666...","(0, 0, 0, 4, 0.2857142857142857, 0.66666666666..."
4,NCT00006110,cancer,"(0, 0, 0, 3, 0.15789473684210525, 0.4285714285...","(0, 0, 0, 5, 0.3333333333333333, 0.71428571428...","(0, 0, 0, 4, 0.25, 0.5714285714285714, 0.34782...","(0, 0, 0, 4, 0.26666666666666666, 0.5714285714..."
...,...,...,...,...,...,...
1688,NCT05204134,diabetes,"(0, 0, 0, 3, 0.2, 0.42857142857142855, 0.27272...","(0, 0, 0, 4, 0.4, 0.5714285714285714, 0.470588...","(0, 0, 0, 2, 0.14285714285714285, 0.2857142857...","(0, 0, 0, 3, 0.2727272727272727, 0.42857142857..."
1689,NCT05289869,obesity,"(1, 0, 0, 3, 0.2727272727272727, 0.33333333333...","(1, 1, 0, 5, 0.4166666666666667, 0.55555555555...","(0, 0, 1, 5, 0.3333333333333333, 0.55555555555...","(1, 1, 0, 3, 0.3, 0.3333333333333333, 0.315789..."
1690,NCT05387889,hypertension,"(0, 0, 0, 4, 0.26666666666666666, 0.5, 0.34782...","(0, 0, 0, 4, 0.36363636363636365, 0.5, 0.42105...","(0, 0, 0, 3, 0.2, 0.375, 0.26086956521739135)","(0, 0, 0, 4, 0.4, 0.5, 0.4444444444444445)"
1691,NCT05451329,hypertension,"(0, 0, 0, 2, 0.2, 0.3333333333333333, 0.25)","(0, 0, 0, 3, 0.375, 0.5, 0.42857142857142855)","(0, 0, 0, 4, 0.4, 0.6666666666666666, 0.5)","(0, 0, 0, 3, 0.375, 0.5, 0.42857142857142855)"


# Workshop Hallucination Record Generation

In [10]:
def transform_to_long_format(repo_hallucination_results):
    """
    Transforms the given DataFrame containing model results into a long format.

    Parameters:
    repo_hallucination_results (pd.DataFrame): The DataFrame with columns for model results.

    Returns:
    pd.DataFrame: Transformed DataFrame in the long format with detailed metrics.
    """
    # Melt and extract tuples into a long format
    long_format = repo_hallucination_results.melt(
        id_vars=['NCTId', 'TrialGroup'],
        value_vars=['gpt4o_zs_gen_hal', 'gpt4o_ts_gen_hal', 'llama3_70b_it_zs_gen_hal', 'llama3_70b_it_ts_gen_hal'],
        var_name='Generation Model',
        value_name='Metrics'
    )

    # Ensure that all entries in the 'Metrics' column are tuples of length 7
    long_format['Metrics'] = long_format['Metrics'].apply(lambda x: x if isinstance(x, tuple) and len(x) == 7 else (None,) * 7)

    # Expand the tuples into their respective columns
    long_format[['Positive Hallucination', 'Negative Hallucination', 'Multi-match Hallucination', 
                 'Correct Matches', 'Precision', 'Recall', 'F1']] = pd.DataFrame(
        long_format['Metrics'].tolist(), index=long_format.index
    )

    # Drop the original 'Metrics' column as it's no longer needed
    long_format.drop(columns=['Metrics'], inplace=True)

    return long_format


In [11]:
long_data = transform_to_long_format(repo_hallucination_results)
long_data.to_csv('workshop_results/CT_Repo_hallucination_results.csv', index=False)
long_data.head(5)

,NCTId,TrialGroup,Generation Model,Positive Hallucination,Negative Hallucination,Multi-match Hallucination,Correct Matches,Precision,Recall,F1
0,NCT00000620,hypertension,gpt4o_zs_gen_hal,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NCT00003901,cancer,gpt4o_zs_gen_hal,0.0,0.0,0.0,4.0,0.222222,0.363636,0.275862
2,NCT00005879,cancer,gpt4o_zs_gen_hal,0.0,0.0,0.0,4.0,0.125000,0.333333,0.181818
3,NCT00005908,cancer,gpt4o_zs_gen_hal,0.0,0.0,0.0,2.0,0.125000,0.333333,0.181818
4,NCT00006110,cancer,gpt4o_zs_gen_hal,0.0,0.0,0.0,3.0,0.157895,0.428571,0.230769


# Score Calculation (Avg)

In [12]:
#calculate average precision, recall and f1 for each model
#average over all examples, save in separate dataframe
adjusted_scores = pd.DataFrame()
adjusted_scores["Metric"] = ["Adjusted Precision", "Adjusted Recall", "Adjusted F1"]
models = ['gpt4o_zs_gen_hal', 'gpt4o_ts_gen_hal', 'llama3_70b_it_zs_gen_hal', 'llama3_70b_it_ts_gen_hal']

#remove None values from the dataframe
repo_hallucination_results = repo_hallucination_results.dropna()
print(repo_hallucination_results.shape)

for model in models:
    adjusted_scores[model] = [repo_hallucination_results[model].apply(lambda x: x[4]).mean(), #precision mean
                              repo_hallucination_results[model].apply(lambda x: x[5]).mean(), #recall mean 
                              repo_hallucination_results[model].apply(lambda x: x[6]).mean()] #f1 mean 

adjusted_scores

(1690, 6)


,Metric,gpt4o_zs_gen_hal,gpt4o_ts_gen_hal,llama3_70b_it_zs_gen_hal,llama3_70b_it_ts_gen_hal
0,Adjusted Precision,0.269433,0.404583,0.324418,0.405222
1,Adjusted Recall,0.505688,0.578875,0.566005,0.563282
2,Adjusted F1,0.331765,0.455705,0.395212,0.448731


# Grouped Score Calculation by TrialGroup

In [13]:
# Group by TrialGroup and calculate mean precision, recall, and F1 scores for each model
grouped_scores = repo_hallucination_results.groupby('TrialGroup').apply(
    lambda x: pd.Series({
        'gpt4o_zero_shot_precision': x['gpt4o_zs_gen_hal'].apply(lambda y: y[4]).mean(),
        'gpt4o_zero_shot_recall': x['gpt4o_zs_gen_hal'].apply(lambda y: y[5]).mean(),
        'gpt4o_zero_shot_f1': x['gpt4o_zs_gen_hal'].apply(lambda y: y[6]).mean(),
        'gpt4o_three_shot_precision': x['gpt4o_ts_gen_hal'].apply(lambda y: y[4]).mean(),
        'gpt4o_three_shot_recall': x['gpt4o_ts_gen_hal'].apply(lambda y: y[5]).mean(),
        'gpt4o_three_shot_f1': x['gpt4o_ts_gen_hal'].apply(lambda y: y[6]).mean(),
        'llama3_zero_shot_precision': x['llama3_70b_it_zs_gen_hal'].apply(lambda y: y[4]).mean(),
        'llama3_zero_shot_recall': x['llama3_70b_it_zs_gen_hal'].apply(lambda y: y[5]).mean(),
        'llama3_zero_shot_f1': x['llama3_70b_it_zs_gen_hal'].apply(lambda y: y[6]).mean(),
        'llama3_three_shot_precision': x['llama3_70b_it_ts_gen_hal'].apply(lambda y: y[4]).mean(),
        'llama3_three_shot_recall': x['llama3_70b_it_ts_gen_hal'].apply(lambda y: y[5]).mean(),
        'llama3_three_shot_f1': x['llama3_70b_it_ts_gen_hal'].apply(lambda y: y[6]).mean(),
    })
).reset_index()

grouped_scores.T

/var/folders/bg/dcwgngc506s6ppbk4kcfrwgr0000gn/T/ipykernel_37892/4147830077.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_scores = repo_hallucination_results.groupby('TrialGroup').apply(


,0,1,2,3,4
TrialGroup,cancer,chronic kidney disease,diabetes,hypertension,obesity
gpt4o_zero_shot_precision,0.21996,0.268356,0.288281,0.300258,0.293059
gpt4o_zero_shot_recall,0.489682,0.519222,0.535703,0.519728,0.462361
gpt4o_zero_shot_f1,0.287552,0.335731,0.356455,0.360806,0.335795
gpt4o_three_shot_precision,0.342373,0.411904,0.430172,0.432014,0.436498
gpt4o_three_shot_recall,0.578878,0.574847,0.596386,0.582289,0.549366
gpt4o_three_shot_f1,0.413368,0.459285,0.480746,0.475652,0.464562
llama3_zero_shot_precision,0.276295,0.343193,0.350255,0.368954,0.31036
llama3_zero_shot_recall,0.538269,0.599814,0.580982,0.613693,0.524402
llama3_zero_shot_f1,0.351203,0.420002,0.420336,0.441081,0.370812
